In [0]:
%pip install azure-eventhub

In [0]:
%restart_python

In [0]:
import json
import random
import time
import uuid
from datetime import datetime, timezone

from azure.eventhub import EventHubProducerClient, EventData

In [0]:
dbutils.widgets.text("num_events", "20", "Number of Events")
dbutils.widgets.text("sleep_seconds", "1", "Seconds Between Events")
dbutils.widgets.dropdown("include_discount_code", "false", ["false", "true"], "Include discount_code")

num_events = int(dbutils.widgets.get("num_events"))
sleep_seconds = float(dbutils.widgets.get("sleep_seconds"))
include_discount_code = dbutils.widgets.get("include_discount_code") == "true"

In [0]:
connection_string = dbutils.secrets.get(
    scope="ecommerce-bronze-scope",
    key="evh-brazilian-ecommerce"
)
eventhub_name = "evh_brazilian_ecommerce"

In [0]:
PRODUCT_IDS = [f"P{i:03d}" for i in range(1, 21)]
CUSTOMER_IDS = [f"C{i:03d}" for i in range(1, 51)]
DISCOUNT_CODES = ["SUMMER25", "WELCOME10", "FLASH15", None]


def generate_order():
    order = {
        "order_id": str(uuid.uuid4())[:8],
        "customer_id": random.choice(CUSTOMER_IDS),
        "product_id": random.choice(PRODUCT_IDS),
        "quantity": random.randint(1, 5),
        "price": round(random.uniform(10.0, 500.0), 2),
        "order_timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    }
    if include_discount_code:
        order["discount_code"] = random.choice(DISCOUNT_CODES)
    return order

In [0]:
producer = EventHubProducerClient.from_connection_string(
    conn_str=connection_string,
    eventhub_name=eventhub_name,
)

sent_count = 0
try:
    for _ in range(num_events):
        batch = producer.create_batch()
        order = generate_order()
        batch.add(EventData(json.dumps(order)))
        producer.send_batch(batch)
        sent_count += 1
        print(f"Sent order {order['order_id']}")
        time.sleep(sleep_seconds)
finally:
    producer.close()

print(f"Done — sent {sent_count} order events (discount_code included: {include_discount_code})")